<a href="https://colab.research.google.com/github/ris2002/Multik30-Eng-De-RNN_LSTM_Comparision/blob/main/RNN_LSTM_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset
import torch
import torch.nn as nn
import torch.optim as optim

dataset = load_dataset("bentrevett/multi30k")
train_data=dataset['train']
test_data=dataset['test']
valid_data=dataset['validation']

In [ ]:
!pip install torchmetrics

In [ ]:
len(train_data)

In [ ]:
len(test_data)

In [ ]:
len(valid_data)

Data Checking

In [ ]:
for item in train_data:
  print(item['en'])
  print(item['de'])
  break





> Data Cleaning



In [ ]:
import re
def clean_data(text,lang):
  text=text.lower().strip()
  if lang=='en':
    text = re.sub(r"[^a-zA-Z\s]", "", text)
  elif lang=='de':
    text=re.sub(r"[^a-zA-ZäöüßÄÖÜ\s]", "", text)
  text = re.sub(r"\s+", " ", text) #cleans extra white space
  return text






In [ ]:
cleaned_results = []
for item in train_data:
  en_cleaned=clean_data(item['en'], 'en')
  de_cleaned=clean_data(item['de'],'de')
  cleaned_results.append({'en': en_cleaned, 'de': de_cleaned})


In [ ]:
for item in cleaned_results[:5]:
    print(f"EN: {item['en']}")
    print(f"DE: {item['de']}")
    print("-" * 10)

In [ ]:
cleaned_results[-1]


Tokenisation Pipeline


In [ ]:
!python -m spacy download de_core_news_sm


In [ ]:
import spacy
from spacy.lang.en import English
from spacy.lang.de import German
nlp_en=English()
nlp_de=German()
def tokenize_eng(cleaned_results:list,en_tokens:list):

  tokenizer=nlp_en.tokenizer
  for text in cleaned_results:
    token_list=[]
    eng_text=text['en']

    doc=tokenizer(eng_text)
    for token in doc:
      token_list.append(token.text)
    en_tokens.append(token_list)

def tokenize_de(cleaned_results:list,de_tokens:list):

  tokenizer=nlp_de.tokenizer
  for text in cleaned_results:
    token_list=[]
    de_text=text['de']

    doc=tokenizer(de_text)
    for token in doc:
      token_list.append(token.text)
    de_tokens.append(token_list)







In [ ]:
en_tokens=[]
de_tokens=[]

In [ ]:
tokenize_eng(cleaned_results,en_tokens)
tokenize_de(cleaned_results,de_tokens)

In [ ]:
en_tokens[:5]

In [ ]:
de_tokens[:5]

Build a vocab dictionary

In [ ]:
from collections import Counter
def build_dictionary(tokens,min_freq=2):
  counter=Counter()
  for text in tokens:
    counter.update(text)
  vocab={"<unk>": 0, "<pad>": 1, "<sos>": 2, "<eos>": 3}
  for word,freq in counter.items():
    if freq>=min_freq:
      vocab[word]=len(vocab)# mapping it to the current size of the 'vocab dict'
  itos={}#index_to_string
  for string,index in vocab.items():
    itos[index]=string
  return vocab,itos

In [ ]:
en_vocab,en_itos=build_dictionary(en_tokens)
de_vocab,de_itos=build_dictionary(de_tokens)

In [ ]:
len(en_vocab)

In [ ]:
en_vocab['<unk>']

In [ ]:
de_vocab['dahinter']

In [ ]:
en_itos[0]

In [ ]:
de_itos[7]

In [ ]:
def mapping(vocab,text):
  mapped_lists=[]
  default=vocab['<unk>']     # there should be no text, only numbers  in the code, previously you put like this default="<unk>"
  for word in text:
    x=vocab.get(word,default)
    mapped_lists.append(x)
  return torch.tensor(mapped_lists) #the embedding layer expects tensors and this func doesnt change the diamention


In [ ]:
mapped_en_texts=[]
for text in en_tokens:
  en_mapped_lists=mapping(en_vocab,text)
  mapped_en_texts.append(en_mapped_lists)
mapped_de_texts=[]
for text in de_tokens:
  de_mapped_lists=mapping(de_vocab,text)
  mapped_de_texts.append(de_mapped_lists)



In [ ]:
mapped_en_texts[:2]

In [ ]:
len(mapped_de_texts)

In [ ]:
mapped_en_texts[:2]

In [ ]:
mapped_de_texts[5].shape

In [ ]:
from torch.nn.utils.rnn import pad_sequence
padded_en_texts=pad_sequence(mapped_en_texts,batch_first=True,padding_value=1)
padded_de_texts=pad_sequence(mapped_de_texts,batch_first=True,padding_value=1)

In [ ]:
padded_en_texts.shape

Pre_Process_Strategy Function from combining steps  based from Above




In [ ]:
import re
def clean_data(text,lang):
  text=text.lower().strip()
  if lang=='en':
    text = re.sub(r"[^a-zA-Z\s]", "", text)
  elif lang=='de':
    text=re.sub(r"[^a-zA-ZäöüßÄÖÜ\s]", "", text)
  text = re.sub(r"\s+", " ", text) #cleans extra white space
  return text

In [ ]:
import spacy
from spacy.lang.en import English
from spacy.lang.de import German
nlp_en=English()
nlp_de=German()
def tokenise_en(cleaned_results:list,en_tokens:list):
  tokeniser=nlp_en.tokenizer
  for text in cleaned_results:
    eng_text=text['en']
    token_list=[]
    doc=tokeniser(eng_text)
    for token in doc:
      token_list.append(token.text)
    en_tokens.append(token_list)
  return en_tokens

def tokenise_de(cleaned_results:list,de_tokens:list):
  tokeniser=nlp_de.tokenizer
  for text in cleaned_results:
    eng_text=text['de']
    token_list=[]
    doc=tokeniser(eng_text)
    for token in doc:
      token_list.append(token.text)
    de_tokens.append(token_list)
  return de_tokens

In [ ]:
from collections import Counter

In [ ]:
def pre_process_strategy_function(data):
  cleaned_results=[]
  en_tokens=[]
  de_tokens=[]
  for item in data:
    eng_cleaned=clean_data(item['en'],'en')
    de_cleaned=clean_data(item['de'],'de')
    cleaned_results.append({'en':eng_cleaned,'de':de_cleaned})
  en_tokens=tokenise_en(cleaned_results,en_tokens)
  de_tokens=tokenise_de(cleaned_results,de_tokens)
  return en_tokens,de_tokens


In [ ]:
#only for training data
from collections import Counter
def build_dictionary(tokens,min_freq=2):
  counter=Counter()
  for text in tokens:
    counter.update(text)
  vocab={"<unk>": 0, "<pad>": 1, "<sos>": 2, "<eos>": 3}
  for word,freq in counter.items():
    if freq>=min_freq:
      vocab[word]=len(vocab)# mapping it to the current size of the 'vocab dict'
  itos={}#index_to_string
  for string,index in vocab.items():
    itos[index]=string
  return vocab,itos


In [ ]:
def mapping(vocab,text,de:bool):
  mapped_lists=[]
  default=vocab['<unk>']
  if de==True:
    for word in text:
      x=vocab.get(word,default)
      mapped_lists.append(x)
    mapped_lists=[2]+mapped_lists+[3]
    return torch.tensor(mapped_lists)

  for word in text:
    x=vocab.get(word,default)
    mapped_lists.append(x)
  return torch.tensor(mapped_lists)

In [ ]:
def mapping_tokens_en(en_vocab,en_tokens):
  mapped_en_texts=[]
  for text in en_tokens:
    en_mapped_lists=mapping(en_vocab,text,de=False)
    mapped_en_texts.append(en_mapped_lists)
  return mapped_en_texts
def mapping_tokens_de(de_vocab,de_tokens):
  mapped_de_texts=[]
  for text in de_tokens:
    de_mapped_lists=mapping(de_vocab,text,de=True)
    mapped_de_texts.append(de_mapped_lists)
  return mapped_de_texts


# **Encoder**

In [ ]:
class Vanilla_RNN_Encoder(nn.Module):
  def __init__(self,input_features,input_vocab_size,input_neurons,num_layers,num_dropout):
    super(Vanilla_RNN_Encoder,self).__init__()
    self.input_neurons = input_neurons
    self.num_layers    = num_layers
    self.dropout=nn.Dropout(num_dropout)
    self.input_embeddings=nn.Embedding(input_vocab_size,input_features)
    self.rnn=nn.RNN(input_features,input_neurons,num_layers=num_layers,dropout=num_dropout,batch_first=True)

  def forward(self,input_tensors,h0=None):
    input_embeddings=self.dropout(self.input_embeddings(input_tensors))
    encoder_out,h_encoder=self.rnn(input_embeddings,h0)
    return h_encoder



**Clarifications regarding Encoder**


1. 'nn.Embedding' only creates a  empty look up table no vals are created, it needs inputs to fill it values, here we have just initialized
2. In  'input_embeddings'  we are trying to convert words into embeddings, each row in this embedding matrix represents each word in the dictionary, row is size of input vocab abd col is richer rep of text(input_embedding_dim)
3. The rnn gives out hidden layer at each steps(a list [h1,h2,...hn]'encoder_out') and the last hidden state(h_encoder)
4. When you use batch_first=True with DataLoader — unsqueeze and squeeze are not needed because DataLoader already adds the batch dimension automatically.




**Internal workings-----**
1. The input tensors should be in a uniform shape. When tensors are created from tokenised lists it would be like [[1,2,3],[16,25,36,43,52],[65,8,98,56,3526]] here we can decode that the rows is the size of the batch and the col are the length of the sequence of the sentence or seq_lenght, therefore thewre shape will be (batchsize x seq_length). this will be padded so that thhe the no. of rows will be uniform.
2. nn.Embedding laayer is the lookup table of size (vocab_size x input_features). For n words a matrix of random numbers are generated (which will be changed during back prop)
ex-

row 0: [0.1,  0.2,  0.3,  0.4]   ← <unk>

row 1: [0.0,  0.0,  0.0,  0.0]   ← <pad>

row 4: [0.3,  0.2, -0.1,  0.5]   ← "I"

row 5: [-0.1, 0.4,  0.3,  0.2]   ← "love"

row 6: [0.2, -0.3,  0.4,  0.1]   ← "dogs

3. When an input tensor is passed through the embedding layer what happens is essentially these rows are feteched
ex-

[[
  
  [0.3,  0.2, -0.1,  0.5],

  [-0.1, 0.4,  0.3,  0.2],

  [0.2, -0.3,  0.4,  0.1] ,

]] so thhe numbers are replaced by the corresponding rows changing its shape to
(batchsize x seq_length x input_features )

4. By default RNN expects input of shape (seq_len x batch x embed_dim) , when batch_first=Ture it accpets inputs of size (batchsize x seq_length x input_features )
5. Inside RNN , it gives out put of shape (batch x seq_len x no_neurons)
6. Data Loader also can be used to pad the data and make it into size (batchsize x seq_length), it can also be used to shuffle data using training. and for h_encoder   = (num_layers x batch x hidden_dim)

7.  In this notebook I have already padded for my reference , but i will still use the dataloader as this is the correct production way.
8. Final logit size after going to Linear layer gives (batch x seq_len x output_vocab_size)




# **Decoder**

In [ ]:
class Vanilla_RNN_Decoder(nn.Module):
  def __init__(self,output_features,output_vocab_size,output_neurons,num_layers,num_dropout):
    super(Vanilla_RNN_Decoder,self).__init__()
    self.output_neurons    = output_neurons
    self.num_layers        = num_layers
    self.output_vocab_size = output_vocab_size
    self.dropout=nn.Dropout(num_dropout)
    self.output_embeddings=nn.Embedding(output_vocab_size,output_features)
    self.rnn=nn.RNN(output_features,output_neurons,num_layers=num_layers,dropout=num_dropout,batch_first=True)
    self.output_logits=nn.Linear(output_neurons,output_vocab_size)

  def forward(self,output_tensors,h0):
    output_embeddings=self.dropout(self.output_embeddings(output_tensors))
    decoder_out, h_decoder=self.rnn(output_embeddings,h0)
    logits=self.output_logits(decoder_out)
    return h_decoder,logits

# ***Seq-Seq Model***

The input to the Seq2Seq model is processed in batches. A batch consists of multiple English and German sentence pairs grouped together. Each sentence is first tokenised and converted to integer indices using the vocabulary dictionary. Since sentences have different lengths, they are padded using pad_sequence to ensure uniform shape of (batch_size x seq_len). This padded tensor is then passed through the embedding layer which replaces each integer index with its corresponding row from the embedding lookup table, changing the shape to (batch_size x seq_len x embed_dim). The encoder processes this full batch simultaneously — producing a context vector which is the final hidden state of shape (num_layers x batch_size x hidden_dim). This context vector is passed to the decoder as the initial hidden state. The decoder then runs in a loop — at each timestep it takes one token from all sentences simultaneously and predicts the next word for every sentence in the batch at once. During training, teacher forcing is used where the actual correct German word is fed as the next input instead of the predicted word, preventing error cascading.



```
batch = 3 sentences

trg = [[<sos>, "ich",  "liebe",  "hunde",  <eos>],   ← sentence 1
       [<sos>, "er",   "läuft",  "schnell", <eos>],  ← sentence 2
       [<sos>, "das",  "ist",    "gut",    <eos>]]    ← sentence 3

shape = (batch=3 x seq_len=5)

trg[0, :] = [<sos>, <sos>, <sos>]   ← one <sos> per sentence
shape = (batch=3,)

t=1 → predict word 1 for all 3 sentences at once
t=2 → predict word 2 for all 3 sentences at once
t=3 → predict word 3 for all 3 sentences at once
```



In [ ]:
class Vanilla_RNN_Seq(nn.Module):
  def __init__(self,decoder,encoder,device):
    super(Vanilla_RNN_Seq,self).__init__()
    self.encoder=encoder
    self.decoder=decoder
    self.device=device
    assert(encoder.input_neurons==decoder.output_neurons)
    assert(encoder.num_layers==decoder.num_layers)
  def forward(self,src,trg,teacher_forcing_ratio):
    h_encoder=self.encoder(src)
    trg_batch_size=trg.shape[0]
    trg_seq_len=trg.shape[1]
    outputs=torch.zeros(trg_batch_size,trg_seq_len,self.decoder.output_vocab_size).to(self.device)
    input=trg[:, 0].unsqueeze(1)

    for t in range(1,trg_seq_len):
      h_decoder,decoder_logits=self.decoder(input,h_encoder)
      h_encoder=h_decoder   # update hidden state for next timestep — decoder output becomes next input
      outputs[:, t]=decoder_logits.squeeze(1)
      teacher_force= random.random()<teacher_forcing_ratio
      top1=decoder_logits.squeeze(1).argmax(1).unsqueeze(1)
      input = trg[:,t].unsqueeze(1) if teacher_force else top1
    return outputs





# LSTM Model

In [ ]:
class LSTM_Encoder(nn.Module):
  def __init__(self,input_features,input_vocab_size,input_neurons,num_layers,num_dropout):
    super(LSTM_Encoder,self).__init__()
    self.input_neurons = input_neurons
    self.num_layers    = num_layers
    self.dropout=nn.Dropout(num_dropout)
    self.input_embeddings=nn.Embedding(input_vocab_size,input_features)
    self.rnn=nn.LSTM(input_features,input_neurons,num_layers=num_layers,dropout=num_dropout,batch_first=True)

  def forward(self,input_tensors,h0=None):
    input_embeddings=self.dropout(self.input_embeddings(input_tensors))
    encoder_out,(h_encoder,cell_encoder)=self.rnn(input_embeddings,h0)
    return h_encoder,cell_encoder

In [ ]:
class LSTM_Decoder(nn.Module):
  def __init__(self,output_features,output_vocab_size,output_neurons,num_layers,num_dropout):
    super(LSTM_Decoder,self).__init__()
    self.output_neurons    = output_neurons
    self.num_layers        = num_layers
    self.output_vocab_size = output_vocab_size
    self.dropout=nn.Dropout(num_dropout)
    self.output_embeddings=nn.Embedding(output_vocab_size,output_features)
    self.rnn=nn.LSTM(output_features,output_neurons,num_layers=num_layers,dropout=num_dropout,batch_first=True)
    self.output_logits=nn.Linear(output_neurons,output_vocab_size)

  def forward(self,output_tensors,h0,cell):
    output_embeddings=self.dropout(self.output_embeddings(output_tensors))
    decoder_out,(h_decoder,cell_decoder)=self.rnn(output_embeddings,(h0, cell))
    logits=self.output_logits(decoder_out)
    return h_decoder,logits,cell_decoder

In [ ]:
class LSTM_Seq(nn.Module):
  def __init__(self,decoder,encoder,device):
    super(LSTM_Seq,self).__init__()
    self.encoder=encoder
    self.decoder=decoder
    self.device=device
    assert(encoder.input_neurons==decoder.output_neurons)
    assert(encoder.num_layers==decoder.num_layers)
  def forward(self,src,trg,teacher_forcing_ratio):
    h_encoder,cell_encoder=self.encoder(src)
    trg_batch_size=trg.shape[0]
    trg_seq_len=trg.shape[1]
    outputs=torch.zeros(trg_batch_size,trg_seq_len,self.decoder.output_vocab_size).to(self.device)
    input=trg[:, 0].unsqueeze(1)

    for t in range(1,trg_seq_len):
      h_decoder,decoder_logits,cell_decoder=self.decoder(input,h_encoder,cell_encoder)
      h_encoder=h_decoder # update hidden state for next timestep — decoder output becomes next input
      outputs[:, t]=decoder_logits.squeeze(1)
      teacher_force= random.random()<teacher_forcing_ratio
      top1=decoder_logits.squeeze(1).argmax(1).unsqueeze(1)
      input = trg[:,t].unsqueeze(1) if teacher_force else top1
    return outputs

Traning and Validation Pipeline

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import logging
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
def training_pipeline(num_epoch, seq_model, optimizer, criterion, batch_size, eng_tensors, de_tensors):
    train_losses = []
    tensor_dataset = TensorDataset(eng_tensors, de_tensors)
    dataloader = DataLoader(tensor_dataset, batch_size=batch_size, shuffle=True)
    for i in range(num_epoch):
        seq_model.train()
        epoch_loss = 0
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            outputs = seq_model(x_batch, y_batch, teacher_forcing_ratio=0.5)
            loss = criterion(outputs.view(-1, len(de_vocab)), y_batch.view(-1))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(dataloader)  # true epoch average
        train_losses.append(avg_loss)
        if i % 10 == 0:
            print(f"Epoch {i}: Loss {avg_loss:.4f}")
    return train_losses


In [ ]:
from torchmetrics.text import SacreBLEUScore
def validation_pipeline(seq_model, eng_tensors, de_tensors, criterion, batch_size):
    seq_model.eval()
    tensor_dataset = TensorDataset(eng_tensors, de_tensors)
    dataloader = DataLoader(tensor_dataset, batch_size=batch_size, shuffle=False)  # no shuffle
    bleu = SacreBLEUScore()  # initialise once, outside loop
    total_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = seq_model(x_batch, y_batch, teacher_forcing_ratio=0)
            loss = criterion(y_pred.view(-1, output_vocab_size), y_batch.view(-1))
            total_loss += loss.item()
            predicted_words = [" ".join([de_itos[idx.item()] for idx in sentence])
                               for sentence in y_pred.argmax(dim=-1)]
            actual_words = [[" ".join([de_itos[idx.item()] for idx in sentence])]
                            for sentence in y_batch]
            bleu(predicted_words, actual_words)
    avg_loss = total_loss / len(dataloader)
    print(f"Validation Loss: {avg_loss:.4f}")
    print(f"BLEU Score: {bleu.compute():.4f}")
    return avg_loss




In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

RNN Training And Testing

In [ ]:
import random

# hyperparameters
INPUT_FEATURES    = 128
OUTPUT_FEATURES   = 128
INPUT_NEURONS     = 128
OUTPUT_NEURONS    = 128
NUM_LAYERS        = 1
NUM_DROPOUT       = 0
NUM_EPOCHS        = 100
BATCH_SIZE        = 32
LEARNING_RATE     = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_vocab_size  = len(en_vocab)
output_vocab_size = len(de_vocab)

# initialise encoder decoder seq2seq
encoder = Vanilla_RNN_Encoder(INPUT_FEATURES, input_vocab_size,
                               INPUT_NEURONS, NUM_LAYERS, NUM_DROPOUT)
decoder = Vanilla_RNN_Decoder(OUTPUT_FEATURES, output_vocab_size,
                               OUTPUT_NEURONS, NUM_LAYERS, NUM_DROPOUT)
model   = Vanilla_RNN_Seq(decoder, encoder, device).to(device)

# optimizer and loss
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=de_vocab['<pad>'])

# count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {count_parameters(model):,} trainable parameters')

# run training
training_pipeline(
    num_epoch     = NUM_EPOCHS,
    seq_model     = model,
    optimizer     = optimizer,
    criterion     = criterion,
    batch_size    = BATCH_SIZE,
    eng_tensors   = padded_en_texts,
    de_tensors    = padded_de_texts
)

In [ ]:
# process validation data
valid_en_tokens, valid_de_tokens = pre_process_strategy_function(valid_data)

# build tensors using same vocab — not new vocab
mapped_en_valid = mapping_tokens_en(en_vocab, valid_en_tokens)
mapped_de_valid = mapping_tokens_de(de_vocab, valid_de_tokens)

# pad
padded_en_texts_valid = pad_sequence(mapped_en_valid, batch_first=True, padding_value=1)
padded_de_texts_valid = pad_sequence(mapped_de_valid, batch_first=True, padding_value=1)

In [ ]:
validation_pipeline(
    seq_model   = model,
    eng_tensors = padded_en_texts_valid,
    de_tensors  = padded_de_texts_valid,
    criterion   = criterion,
    batch_size  = BATCH_SIZE
)

LSTM Training And Testing

In [ ]:
import random

# hyperparameters
INPUT_FEATURES    = 128
OUTPUT_FEATURES   = 128
INPUT_NEURONS     = 128
OUTPUT_NEURONS    = 128
NUM_LAYERS        = 1
NUM_DROPOUT       = 0
NUM_EPOCHS        = 100
BATCH_SIZE        = 32
LEARNING_RATE     = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_vocab_size  = len(en_vocab)
output_vocab_size = len(de_vocab)

# initialise encoder decoder seq2seq
encoder = LSTM_Encoder(INPUT_FEATURES, input_vocab_size,
                               INPUT_NEURONS, NUM_LAYERS, NUM_DROPOUT)
decoder = LSTM_Decoder(OUTPUT_FEATURES, output_vocab_size,
                               OUTPUT_NEURONS, NUM_LAYERS, NUM_DROPOUT)
model   = LSTM_Seq(decoder, encoder, device).to(device)

# optimizer and loss
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=de_vocab['<pad>'])

# count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {count_parameters(model):,} trainable parameters')

# run training
training_pipeline(
    num_epoch     = NUM_EPOCHS,
    seq_model     = model,
    optimizer     = optimizer,
    criterion     = criterion,
    batch_size    = BATCH_SIZE,
    eng_tensors   = padded_en_texts,
    de_tensors    = padded_de_texts
)

In [ ]:
# process validation data
valid_en_tokens, valid_de_tokens = pre_process_strategy_function(valid_data)

# build tensors using same vocab — not new vocab
mapped_en_valid = mapping_tokens_en(en_vocab, valid_en_tokens)
mapped_de_valid = mapping_tokens_de(de_vocab, valid_de_tokens)

# pad
padded_en_texts_valid = pad_sequence(mapped_en_valid, batch_first=True, padding_value=1)
padded_de_texts_valid = pad_sequence(mapped_de_valid, batch_first=True, padding_value=1)

In [ ]:
validation_pipeline(
    seq_model   = model,
    eng_tensors = padded_en_texts_valid,
    de_tensors  = padded_de_texts_valid,
    criterion   = criterion,
    batch_size  = BATCH_SIZE
)